In [1]:
# Install required libraries
%pip install opencv-python
%pip install mediapipe
%pip install pyautogui
%pip install pynput

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
# Import core computer vision and GUI automation libraries
import cv2
import numpy as np
import util
import random
import mediapipe as mp
from mediapipe.tasks import python
import pyautogui
from pynput.mouse import Button, Controller
from mediapipe.tasks.python import vision

In [3]:
import time

# Initialize screen dimensions and mouse controller
screen_width, screen_height = pyautogui.size()
mouse = Controller()

# Cooldown tracking variables
last_action_time = 0
COOLDOWN_DELAY = 0.5  # Time in seconds to wait between clicks

# Initialize MediaPipe HandLandmarker Task for real-time video stream processing
base_options = python.BaseOptions(model_asset_path='hand_landmarker.task')

options = vision.HandLandmarkerOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.VIDEO,  # Optimized for continuous video streams
    num_hands=1,
    min_hand_detection_confidence=0.7,
    min_hand_presence_confidence=0.7,
    min_tracking_confidence=0.7
)

# Create detector instance
detector = vision.HandLandmarker.create_from_options(options)

In [4]:
def find_finger_tip(processed):
    """
    Extracts the normalized coordinates of the index finger tip from detector results.
    Landmark 8 corresponds to the index finger tip in MediaPipe's hand model.
    """
    if processed.hand_landmarks:
        hand_landmarks = processed.hand_landmarks[0]
        return hand_landmarks[8]  # Index 8 is INDEX_FINGER_TIP landmark

    return None

In [5]:
def move_mouse(index_finger_tip):
    """
    Maps normalized finger tip coordinates (0.0 to 1.0) to screen resolution 
    and updates the system cursor position using PyAutoGUI.
    """
    if index_finger_tip is not None:
        x = int(index_finger_tip.x * screen_width)
        y = int(index_finger_tip.y * screen_height)
        pyautogui.moveTo(x, y)

In [6]:
# Gesture condition evaluation functions using joint angles and landmark distances

def is_left_click(landmarks_list, thumb_index_distance):
    """Triggers Left Click based on index finger flexion and thumb separation."""
    return (util.get_angle(landmarks_list[5], landmarks_list[6], landmarks_list[8]) < 50 and 
            util.get_angle(landmarks_list[9], landmarks_list[10], landmarks_list[12]) > 90 and 
            thumb_index_distance > 50)

def is_right_click(landmarks_list, thumb_index_distance):
    """Triggers Right Click based on middle finger flexion and thumb separation."""
    return (util.get_angle(landmarks_list[5], landmarks_list[6], landmarks_list[8]) > 90 and 
            util.get_angle(landmarks_list[9], landmarks_list[10], landmarks_list[12]) < 50 and 
            thumb_index_distance > 50)

def is_double_click(landmarks_list, thumb_index_distance):
    """Triggers Double Click when index and middle fingers are simultaneously flexed."""
    return (util.get_angle(landmarks_list[5], landmarks_list[6], landmarks_list[8]) < 50 and 
            util.get_angle(landmarks_list[9], landmarks_list[10], landmarks_list[12]) < 50 and 
            thumb_index_distance > 50)

def is_screenshot(landmarks_list, thumb_index_distance):
    """Triggers a Screenshot action when hand gesture criteria are met."""
    return (util.get_angle(landmarks_list[5], landmarks_list[6], landmarks_list[8]) < 50 and 
            util.get_angle(landmarks_list[9], landmarks_list[10], landmarks_list[12]) > 90 and 
            util.get_angle(landmarks_list[13], landmarks_list[14], landmarks_list[16]) > 90 and 
            util.get_angle(landmarks_list[17], landmarks_list[18], landmarks_list[20]) > 90 and
            thumb_index_distance < 50)

In [7]:
def detect_gestures(frame, landmarks_list, processed):
    """
    Evaluates detected hand landmarks, identifies gestures, and executes
    corresponding mouse/keyboard actions with a cooldown delay for clicks.
    """
    global last_action_time

    if len(landmarks_list) >= 21:
        index_finger_tip = find_finger_tip(processed)
        thumb_index_distance = util.get_distance([landmarks_list[4], landmarks_list[5]])

        # Mouse movement (Continuous tracking, no cooldown needed)
        if thumb_index_distance < 50 and util.get_angle(landmarks_list[5], landmarks_list[6], landmarks_list[8]) > 90:
            move_mouse(index_finger_tip)

        # Clicks & Actions (Protected by cooldown delay)
        else:
            current_time = time.time()
            if current_time - last_action_time > COOLDOWN_DELAY:

                if is_left_click(landmarks_list, thumb_index_distance):
                    mouse.press(Button.left)
                    mouse.release(Button.left)
                    cv2.putText(frame, "Left Click", (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
                    last_action_time = current_time

                elif is_right_click(landmarks_list, thumb_index_distance):
                    mouse.press(Button.right)
                    mouse.release(Button.right)
                    cv2.putText(frame, "Right Click", (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
                    last_action_time = current_time

                elif is_double_click(landmarks_list, thumb_index_distance):
                    pyautogui.doubleClick()
                    cv2.putText(frame, "Double Click", (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
                    last_action_time = current_time

                elif is_screenshot(landmarks_list, thumb_index_distance):        
                    im1 = pyautogui.screenshot()
                    label = random.randint(1, 1000)
                    im1.save(f'screenshot_by_joo_mouse_{label}.png')
                    cv2.putText(frame, "Screenshot Taken", (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
                    last_action_time = current_time

In [8]:
def main():
    """Main execution loop for video capture, MediaPipe inference, and UI rendering."""
    cap = cv2.VideoCapture(0)

    try:
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break

            # Mirror frame horizontally for intuitive self-tracking
            frame = cv2.flip(frame, 1)
            frameRGB = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            
            # Convert frame to MediaPipe Image format
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frameRGB)
            timestamp_ms = int(cv2.getTickCount() / cv2.getTickFrequency() * 1000)

            # Perform hand detection for video stream
            processed = detector.detect_for_video(mp_image, timestamp_ms)
            landmarks_list = []

            # Process hand landmarks if present
            if processed.hand_landmarks:
                hand_landmarks = processed.hand_landmarks[0]

                for lm in hand_landmarks:
                    landmarks_list.append((lm.x, lm.y))

                # Draw landmark circles on OpenCV output frame
                h, w, _ = frame.shape
                for lm in hand_landmarks:
                    cv2.circle(frame, (int(lm.x * w), int(lm.y * h)), 5, (0, 255, 0), -1)

                print("Landmarks:", landmarks_list)

            # Run gesture classification logic
            detect_gestures(frame, landmarks_list, processed)

            # Display feed
            cv2.imshow('Joo Mouse', frame)
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break

    finally:
        # Clean up video capture resources and close open windows
        cap.release()
        cv2.destroyAllWindows()

if __name__ == "__main__":
    main()

Landmarks: [(0.4844962954521179, 1.0096080303192139), (0.5440300107002258, 0.9284458160400391), (0.5783563256263733, 0.8335171341896057), (0.5972273349761963, 0.7680330872535706), (0.6184637546539307, 0.7111173272132874), (0.5110188722610474, 0.7285906076431274), (0.5218111276626587, 0.6332687735557556), (0.5655377507209778, 0.6250449419021606), (0.6022211313247681, 0.6454206705093384), (0.476604163646698, 0.7475889325141907), (0.5570829510688782, 0.7163034081459045), (0.598253607749939, 0.7732020020484924), (0.6104324460029602, 0.827485203742981), (0.4524609446525574, 0.787273108959198), (0.5441636443138123, 0.7873587012290955), (0.570239245891571, 0.8580611944198608), (0.5702202916145325, 0.9086416363716125), (0.43772727251052856, 0.840681791305542), (0.5128849744796753, 0.8370710611343384), (0.5440567135810852, 0.8819743990898132), (0.5529587864875793, 0.9191975593566895)]
Landmarks: [(0.48572635650634766, 0.9601983428001404), (0.540595531463623, 0.8763532042503357), (0.569314062595